### Data Reading

In [1]:
import os
import sys
import glob

# 1. Add base PySpark path
sys.path.insert(0, '/opt/spark/python')

# 2. Dynamically locate and add py4j zip file inside /opt/spark/python/lib/
py4j_zip = glob.glob('/opt/spark/python/lib/py4j-*.zip')
if py4j_zip:
    sys.path.insert(0, py4j_zip[0])

# 3. Now import and start Spark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PySpark-Practice") \
    .master("spark://spark-master:7077") \
    .config("spark.driver.host", "jupyter-local") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.driver.port", "29200") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/25 02:35:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df = spark.read.format('csv') \
    .option("header", True) \
    .option("inferSchema", True) \
    .load("/opt/spark-apps/BigMart Sales.csv")

In [3]:
df.show()
# df.display() for databricks

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138|
|          DRC01|       5.92|         Regular|    0.019278216|         Soft Drinks| 48.2692|           OUT018|                     2009|     Medium|              Tier 3|Superma

In [4]:
df_json = spark.read.format('json') \
            .option('inferSchema', True) \
            .option('header', True) \
            .option('multiLine', False) \
            .load('/opt/spark-apps/drivers.json')

In [5]:
df_json.show()

+----+----------+--------+----------+--------------------+-----------+------+--------------------+
|code|       dob|driverId| driverRef|                name|nationality|number|                 url|
+----+----------+--------+----------+--------------------+-----------+------+--------------------+
| HAM|1985-01-07|       1|  hamilton|   {Lewis, Hamilton}|    British|    44|http://en.wikiped...|
| HEI|1977-05-10|       2|  heidfeld|    {Nick, Heidfeld}|     German|    \N|http://en.wikiped...|
| ROS|1985-06-27|       3|   rosberg|     {Nico, Rosberg}|     German|     6|http://en.wikiped...|
| ALO|1981-07-29|       4|    alonso|  {Fernando, Alonso}|    Spanish|    14|http://en.wikiped...|
| KOV|1981-10-19|       5|kovalainen|{Heikki, Kovalainen}|    Finnish|    \N|http://en.wikiped...|
| NAK|1985-01-11|       6|  nakajima|  {Kazuki, Nakajima}|   Japanese|    \N|http://en.wikiped...|
| BOU|1979-02-28|       7|  bourdais|{Sébastien, Bourd...|     French|    \N|http://en.wikiped...|
| RAI|1979

### Schema Definition

In [6]:
df.printSchema()

root
 |-- Item_Identifier: string (nullable = true)
 |-- Item_Weight: double (nullable = true)
 |-- Item_Fat_Content: string (nullable = true)
 |-- Item_Visibility: double (nullable = true)
 |-- Item_Type: string (nullable = true)
 |-- Item_MRP: double (nullable = true)
 |-- Outlet_Identifier: string (nullable = true)
 |-- Outlet_Establishment_Year: integer (nullable = true)
 |-- Outlet_Size: string (nullable = true)
 |-- Outlet_Location_Type: string (nullable = true)
 |-- Outlet_Type: string (nullable = true)
 |-- Item_Outlet_Sales: double (nullable = true)



In [7]:
my_ddl_schema = '''
                    Item_Identifier STRING,
                    Item_Weight STRING,
                    Item_Fat_Content STRING,
                    Item_Visibility DOUBLE,
                    Item_Type STRING,
                    Item_MRP DOUBLE,
                    Outlet_Identifier STRING,
                    Outlet_Establishment_Year INT,
                    Outlet_Size STRING,
                    Outlet_Location_Type STRING,
                    Outlet_Type STRING,
                    Item_Outlet_Sales DOUBLE
                '''
# Changes Item_Weight to String type

In [8]:
df = spark.read.format('csv') \
                .schema(my_ddl_schema) \
                .option('header', True) \
                .load('/opt/spark-apps/BigMart Sales.csv')

In [9]:
df.printSchema()

root
 |-- Item_Identifier: string (nullable = true)
 |-- Item_Weight: string (nullable = true)
 |-- Item_Fat_Content: string (nullable = true)
 |-- Item_Visibility: double (nullable = true)
 |-- Item_Type: string (nullable = true)
 |-- Item_MRP: double (nullable = true)
 |-- Outlet_Identifier: string (nullable = true)
 |-- Outlet_Establishment_Year: integer (nullable = true)
 |-- Outlet_Size: string (nullable = true)
 |-- Outlet_Location_Type: string (nullable = true)
 |-- Outlet_Type: string (nullable = true)
 |-- Item_Outlet_Sales: double (nullable = true)



### StructType() Schema

In [10]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [11]:
my_struct_schema = StructType([
    StructField('Item_Identifier', StringType(), True),
    StructField('Item_Weight', StringType(), True),
    StructField('Item_Fat_Content', StringType(), True),
    StructField('Item_Visibility', StringType(), True),
    StructField('Item_Type', StringType(), True),
    StructField('Item_MRP', StringType(), True),
    StructField('Outlet_Identifier', StringType(), True),
    StructField('Outlet_Establishment_Year', StringType(), True),
    StructField('Outlet_Size', StringType(), True),
    StructField('Outlet_Location_Type', StringType(), True),
    StructField('Outlet_Type', StringType(), True),
    StructField('Item_Outlet_Sales', StringType(), True),
])

In [12]:
df = spark.read.format('csv') \
                .schema(my_struct_schema) \
                .option('header', True) \
                .load('/opt/spark-apps/BigMart Sales.csv')

In [13]:
df.printSchema()

root
 |-- Item_Identifier: string (nullable = true)
 |-- Item_Weight: string (nullable = true)
 |-- Item_Fat_Content: string (nullable = true)
 |-- Item_Visibility: string (nullable = true)
 |-- Item_Type: string (nullable = true)
 |-- Item_MRP: string (nullable = true)
 |-- Outlet_Identifier: string (nullable = true)
 |-- Outlet_Establishment_Year: string (nullable = true)
 |-- Outlet_Size: string (nullable = true)
 |-- Outlet_Location_Type: string (nullable = true)
 |-- Outlet_Type: string (nullable = true)
 |-- Item_Outlet_Sales: string (nullable = true)



### SELECT

In [14]:
df.select('Item_Identifier', 'Item_Weight', 'Item_Fat_Content').show()

+---------------+-----------+----------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|
+---------------+-----------+----------------+
|          FDA15|        9.3|         Low Fat|
|          DRC01|       5.92|         Regular|
|          FDN15|       17.5|         Low Fat|
|          FDX07|       19.2|         Regular|
|          NCD19|       8.93|         Low Fat|
|          FDP36|     10.395|         Regular|
|          FDO10|      13.65|         Regular|
|          FDP10|       NULL|         Low Fat|
|          FDH17|       16.2|         Regular|
|          FDU28|       19.2|         Regular|
|          FDY07|       11.8|         Low Fat|
|          FDA03|       18.5|         Regular|
|          FDX32|       15.1|         Regular|
|          FDS46|       17.6|         Regular|
|          FDF32|      16.35|         Low Fat|
|          FDP49|          9|         Regular|
|          NCB42|       11.8|         Low Fat|
|          FDP49|          9|         Regular|
|          DR

In [15]:
df.select(col('Item_Identifier'), col('Item_Weight'), col('Item_Fat_Content')).show()

+---------------+-----------+----------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|
+---------------+-----------+----------------+
|          FDA15|        9.3|         Low Fat|
|          DRC01|       5.92|         Regular|
|          FDN15|       17.5|         Low Fat|
|          FDX07|       19.2|         Regular|
|          NCD19|       8.93|         Low Fat|
|          FDP36|     10.395|         Regular|
|          FDO10|      13.65|         Regular|
|          FDP10|       NULL|         Low Fat|
|          FDH17|       16.2|         Regular|
|          FDU28|       19.2|         Regular|
|          FDY07|       11.8|         Low Fat|
|          FDA03|       18.5|         Regular|
|          FDX32|       15.1|         Regular|
|          FDS46|       17.6|         Regular|
|          FDF32|      16.35|         Low Fat|
|          FDP49|          9|         Regular|
|          NCB42|       11.8|         Low Fat|
|          FDP49|          9|         Regular|
|          DR

### ALIAS

In [16]:
df.select(col('Item_Identifier').alias('ItemID')).show()

+------+
|ItemID|
+------+
| FDA15|
| DRC01|
| FDN15|
| FDX07|
| NCD19|
| FDP36|
| FDO10|
| FDP10|
| FDH17|
| FDU28|
| FDY07|
| FDA03|
| FDX32|
| FDS46|
| FDF32|
| FDP49|
| NCB42|
| FDP49|
| DRI11|
| FDU02|
+------+
only showing top 20 rows


### FILTER

#### Scenario 1

In [17]:
df.filter(col('Item_Fat_Content')=='Regular').show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|          DRC01|       5.92|         Regular|    0.019278216|         Soft Drinks| 48.2692|           OUT018|                     2009|     Medium|              Tier 3|Supermarket Type2|         443.4228|
|          FDX07|       19.2|         Regular|              0|Fruits and Vegeta...| 182.095|           OUT010|                     1998|       NULL|              Tier 3|    Gro

#### Scenario 2

In [18]:
df.filter( (col('Item_Type')=='Soft Drinks') & (col('Item_Weight')<10.0) ).show()

+---------------+-----------+----------------+---------------+-----------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|  Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|
+---------------+-----------+----------------+---------------+-----------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|          DRC01|       5.92|         Regular|    0.019278216|Soft Drinks| 48.2692|           OUT018|                     2009|     Medium|              Tier 3|Supermarket Type2|         443.4228|
|          DRZ11|       8.85|         Regular|    0.113123893|Soft Drinks|122.5388|           OUT018|                     2009|     Medium|              Tier 3|Supermarket Type2|        1609.9044|
|          DRF4

#### Scenario 3

In [19]:
df.filter( (col('Outlet_Size').isNull()) \
          & (col('Outlet_Location_Type').isin('Tier 1', 'Tier 2')) ).show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|          FDH17|       16.2|         Regular|    0.016687114|        Frozen Foods| 96.9726|           OUT045|                     2002|       NULL|              Tier 2|Supermarket Type1|        1076.5986|
|          FDU28|       19.2|         Regular|     0.09444959|        Frozen Foods|187.8214|           OUT017|                     2007|       NULL|              Tier 2|Superma

### withColumnRenamed

In [20]:
df.withColumnRenamed('Item_Weight', 'Item_Wt').show()

+---------------+-------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|Item_Identifier|Item_Wt|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|
+---------------+-------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|          FDA15|    9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138|
|          DRC01|   5.92|         Regular|    0.019278216|         Soft Drinks| 48.2692|           OUT018|                     2009|     Medium|              Tier 3|Supermarket Type2|         

### withColumn

#### Scenario 1

In [21]:
df = df.withColumn('flag', lit('new')) # lit to put constant value

In [22]:
df.show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138| new|
|          DRC01|       5.92|         Regular|    0.019278216|         Soft Drinks| 48.2692|           OUT018|                     2009|     Medium|        

In [23]:
df = df.withColumn('Item_Weight', col('Item_Weight').cast(DoubleType()))

In [24]:
df.withColumn('multiply', col('Item_Weight')*col('Item_MRP')).show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+------------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag|          multiply|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+------------------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138| new|2323.2255600000003|
|          DRC01|       5.92|         Regular|    0.019278216|         Soft Drin

#### Scenario 2

In [25]:
# regexp_replace is used to replace column values
df.withColumn('Item_Fat_Content', regexp_replace(col('Item_Fat_Content'),"Regular","Reg")) \
    .withColumn('Item_Fat_Content', regexp_replace(col('Item_Fat_Content'),"Low Fat","LF")) \
    .show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+
|          FDA15|        9.3|              LF|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138| new|
|          DRC01|       5.92|             Reg|    0.019278216|         Soft Drinks| 48.2692|           OUT018|                     2009|     Medium|        

### Type Casting

In [26]:
df = df.withColumn('Item_Weight', col('Item_Weight').cast(StringType()))

In [27]:
df.printSchema()

root
 |-- Item_Identifier: string (nullable = true)
 |-- Item_Weight: string (nullable = true)
 |-- Item_Fat_Content: string (nullable = true)
 |-- Item_Visibility: string (nullable = true)
 |-- Item_Type: string (nullable = true)
 |-- Item_MRP: string (nullable = true)
 |-- Outlet_Identifier: string (nullable = true)
 |-- Outlet_Establishment_Year: string (nullable = true)
 |-- Outlet_Size: string (nullable = true)
 |-- Outlet_Location_Type: string (nullable = true)
 |-- Outlet_Type: string (nullable = true)
 |-- Item_Outlet_Sales: string (nullable = true)
 |-- flag: string (nullable = false)



In [28]:
# Cleanup
df = df.withColumn('Item_Weight', col('Item_Weight').cast(DoubleType()))

### Sort

#### Scenario 1

In [29]:
df.sort(col('Item_Weight').desc()).show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+
|          FDC02|      21.35|         Low Fat|    0.068809463|              Canned|258.5278|           OUT035|                     2004|      Small|              Tier 2|Supermarket Type1|         5206.556| new|
|          FDC02|      21.35|         Low Fat|    0.069102831|              Canned|259.9278|           OUT018|                     2009|     Medium|        

#### Scenario 2

In [30]:
df.sort(col('Item_Visibility').asc()).show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+
|          DRB48|      16.75|         Regular|              0|         Soft Drinks| 39.3822|           OUT046|                     1997|      Small|              Tier 1|Supermarket Type1|         353.5398| new|
|          FDS32|      17.75|         Regular|              0|Fruits and Vegeta...|139.9838|           OUT045|                     2002|       NULL|        

#### Scenario 3

In [31]:
df.sort(['Item_Weight', 'Item_Visibility'], ascending = [0, 0]).show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+
|          FDR07|      21.35|         Low Fat|    0.130127365|Fruits and Vegeta...| 96.2094|           OUT010|                     1998|       NULL|              Tier 3|    Grocery Store|         190.4188| new|
|          FDC02|      21.35|         Low Fat|    0.115194717|              Canned|258.3278|           OUT010|                     1998|       NULL|        

#### Scenario 4

In [32]:
df.sort(['Item_Weight', 'Item_Visibility'], ascending=[0, 1]).show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+
|          FDC02|      21.35|         Low Fat|    0.068765205|              Canned|260.4278|           OUT013|                     1987|       High|              Tier 3|Supermarket Type1|        3644.5892| new|
|          FDC02|      21.35|         Low Fat|    0.068809463|              Canned|258.5278|           OUT035|                     2004|      Small|        

### Limit

In [33]:
df.limit(10).show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138| new|
|          DRC01|       5.92|         Regular|    0.019278216|         Soft Drinks| 48.2692|           OUT018|                     2009|     Medium|        

### Drop

#### Scenario 1

In [34]:
df.drop('Item_Visibility').show()

+---------------+-----------+----------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+
|Item_Identifier|Item_Weight|Item_Fat_Content|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag|
+---------------+-----------+----------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+
|          FDA15|        9.3|         Low Fat|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138| new|
|          DRC01|       5.92|         Regular|         Soft Drinks| 48.2692|           OUT018|                     2009|     Medium|              Tier 3|Supermarket Type2|         443.4228| new|
|          FDN15|       1

#### Scenario 2

In [35]:
df.drop('Item_Visibility', 'Item_Type').show()

+---------------+-----------+----------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag|
+---------------+-----------+----------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+
|          FDA15|        9.3|         Low Fat|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138| new|
|          DRC01|       5.92|         Regular| 48.2692|           OUT018|                     2009|     Medium|              Tier 3|Supermarket Type2|         443.4228| new|
|          FDN15|       17.5|         Low Fat| 141.618|           OUT049|                     1999|     Medium|              Tier 

### DROP_DUPLICATES

#### Scenario 1

In [36]:
df.dropDuplicates().show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+
|          FDR02|       16.7|         Low Fat|    0.022047313|               Dairy|110.2886|           OUT013|                     1987|       High|              Tier 3|Supermarket Type1|        1890.2062| new|
|          FDA28|       16.1|         Regular|    0.047801883|        Frozen Foods|127.4362|           OUT046|                     1997|      Small|        

#### Scenario 2

In [37]:
df.drop_duplicates(subset=['Item_Type']).show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+
|          FDP36|     10.395|         Regular|              0|        Baking Goods| 51.4008|           OUT018|                     2009|     Medium|              Tier 3|Supermarket Type2|         556.6088| new|
|          FDO23|      17.85|         Low Fat|              0|              Breads| 93.1436|           OUT045|                     2002|       NULL|        

In [38]:
df.distinct().show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+
|          FDR02|       16.7|         Low Fat|    0.022047313|               Dairy|110.2886|           OUT013|                     1987|       High|              Tier 3|Supermarket Type1|        1890.2062| new|
|          FDA28|       16.1|         Regular|    0.047801883|        Frozen Foods|127.4362|           OUT046|                     1997|      Small|        

### UNION and UNION BY NAME

#### Preparing Dataframes

In [39]:
data1 = [
    ('1', 'kad'),
    ('2', 'sid')
]
schema1 = 'id STRING, name STRING'

df1 = spark.createDataFrame(data1, schema1)

data2 = [
    ('3', 'rahul'),
    ('4', 'jas')
]
schema2 = 'id STRING, name STRING'

df2 = spark.createDataFrame(data2, schema2)

In [40]:
df1.show()

+---+----+
| id|name|
+---+----+
|  1| kad|
|  2| sid|
+---+----+



In [41]:
df2.show()

+---+-----+
| id| name|
+---+-----+
|  3|rahul|
|  4|  jas|
+---+-----+



### Union

In [42]:
df1.union(df2).show()

+---+-----+
| id| name|
+---+-----+
|  1|  kad|
|  2|  sid|
|  3|rahul|
|  4|  jas|
+---+-----+



In [43]:
data1 = [
    ('kad', '1'),
    ('sid', '2')
]
schema1 = 'name STRING, id STRING'

df1 = spark.createDataFrame(data1, schema1)

df1.show()

+----+---+
|name| id|
+----+---+
| kad|  1|
| sid|  2|
+----+---+



In [44]:
df1.union(df2).show()

+----+-----+
|name|   id|
+----+-----+
| kad|    1|
| sid|    2|
|   3|rahul|
|   4|  jas|
+----+-----+



### Union by Name

In [45]:
df1.unionByName(df2).show()

+-----+---+
| name| id|
+-----+---+
|  kad|  1|
|  sid|  2|
|rahul|  3|
|  jas|  4|
+-----+---+



### String Functions

In [46]:
df.select(initcap('Item_Type')).show()

+--------------------+
|  initcap(Item_Type)|
+--------------------+
|               Dairy|
|         Soft Drinks|
|                Meat|
|Fruits And Vegeta...|
|           Household|
|        Baking Goods|
|         Snack Foods|
|         Snack Foods|
|        Frozen Foods|
|        Frozen Foods|
|Fruits And Vegeta...|
|               Dairy|
|Fruits And Vegeta...|
|         Snack Foods|
|Fruits And Vegeta...|
|           Breakfast|
|  Health And Hygiene|
|           Breakfast|
|         Hard Drinks|
|               Dairy|
+--------------------+
only showing top 20 rows


In [47]:
df.select(lower('Item_Type')).show()

+--------------------+
|    lower(Item_Type)|
+--------------------+
|               dairy|
|         soft drinks|
|                meat|
|fruits and vegeta...|
|           household|
|        baking goods|
|         snack foods|
|         snack foods|
|        frozen foods|
|        frozen foods|
|fruits and vegeta...|
|               dairy|
|fruits and vegeta...|
|         snack foods|
|fruits and vegeta...|
|           breakfast|
|  health and hygiene|
|           breakfast|
|         hard drinks|
|               dairy|
+--------------------+
only showing top 20 rows


In [48]:
df.select(upper('Item_Type').alias('Upper_Item_Type')).show()

+--------------------+
|     Upper_Item_Type|
+--------------------+
|               DAIRY|
|         SOFT DRINKS|
|                MEAT|
|FRUITS AND VEGETA...|
|           HOUSEHOLD|
|        BAKING GOODS|
|         SNACK FOODS|
|         SNACK FOODS|
|        FROZEN FOODS|
|        FROZEN FOODS|
|FRUITS AND VEGETA...|
|               DAIRY|
|FRUITS AND VEGETA...|
|         SNACK FOODS|
|FRUITS AND VEGETA...|
|           BREAKFAST|
|  HEALTH AND HYGIENE|
|           BREAKFAST|
|         HARD DRINKS|
|               DAIRY|
+--------------------+
only showing top 20 rows


### Date Functions

#### Current_Date

In [49]:
df = df.withColumn('curr_date', current_date())

df.show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag| curr_date|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138| new|2026-07-25|
|          DRC01|       5.92|         Regular|    0.019278216|         Soft Drinks| 48.2692|           OUT018|  

#### Date_Add()

In [50]:
df = df.withColumn('week_after', date_add('curr_date', 7))
df.show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag| curr_date|week_after|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138| new|2026-07-25|2026-08-01|
|          DRC01|       5.92|         Regular|    0.019278216|      

#### Date_Sub()

In [51]:
df.withColumn('week_before', date_sub('curr_date', 7)).show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag| curr_date|week_after|week_before|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138| new|2026-07-25|2026-08-01| 2026-07-18|
|          DRC01|   

In [52]:
df = df.withColumn('week_before', date_add('curr_date', -7))
df.show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag| curr_date|week_after|week_before|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138| new|2026-07-25|2026-08-01| 2026-07-18|
|          DRC01|   

### DateDIFF

In [53]:
df = df.withColumn('datediff', datediff('curr_date', 'week_before'))
df.show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag| curr_date|week_after|week_before|datediff|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138| new|2026-07-25|2026-08-01| 2026-

### Date_Format()

In [54]:
df = df.withColumn('week_before', date_format(col('week_before'), 'dd-MM-yyyy'))
df.show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag| curr_date|week_after|week_before|datediff|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138| new|2026-07-25|2026-08-01| 18-07

### Handling Nulls

#### Dropping Nulls

In [55]:
df.dropna('all').show() # drop records that have null in all columns

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag| curr_date|week_after|week_before|datediff|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138| new|2026-07-25|2026-08-01| 18-07

In [56]:
df.dropna('any').show() # Drop records that have null value at any column

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag| curr_date|week_after|week_before|datediff|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138| new|2026-07-25|2026-08-01| 18-07

In [57]:
df.dropna(subset=['Outlet_Size']).show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag| curr_date|week_after|week_before|datediff|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138| new|2026-07-25|2026-08-01| 18-07

#### Filling Nulls

In [58]:
df.fillna('NotAvailable').show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+------------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year| Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag| curr_date|week_after|week_before|datediff|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+------------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|      Medium|              Tier 1|Supermarket Type1|         3735.138| new|2026-07-25|2026-08-01| 1

In [59]:
df.fillna('NotAvailable', subset=['Outlet_Size']).show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+------------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year| Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag| curr_date|week_after|week_before|datediff|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+------------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|      Medium|              Tier 1|Supermarket Type1|         3735.138| new|2026-07-25|2026-08-01| 1

### SPLIT and Indexing

In [60]:
df.withColumn('Outlet_Type', split('Outlet_Type', ' ')).show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+--------------------+-----------------+----+----------+----------+-----------+--------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|         Outlet_Type|Item_Outlet_Sales|flag| curr_date|week_after|week_before|datediff|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+--------------------+-----------------+----+----------+----------+-----------+--------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|[Supermarket, Type1]|         3735.138| new|2026-07-25|2026-

In [61]:
df.withColumn('Outlet_Type', split('Outlet_Type', ' ')[1]).show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------+-----------------+----+----------+----------+-----------+--------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|Outlet_Type|Item_Outlet_Sales|flag| curr_date|week_after|week_before|datediff|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------+-----------------+----+----------+----------+-----------+--------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|      Type1|         3735.138| new|2026-07-25|2026-08-01| 18-07-2026|       7|
|       

### EXPLODE

In [62]:
df_exp = df.withColumn('Outlet_Type', split('Outlet_Type', ' '))

In [63]:
df_exp.show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+--------------------+-----------------+----+----------+----------+-----------+--------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|         Outlet_Type|Item_Outlet_Sales|flag| curr_date|week_after|week_before|datediff|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+--------------------+-----------------+----+----------+----------+-----------+--------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|[Supermarket, Type1]|         3735.138| new|2026-07-25|2026-

In [64]:
df_exp.withColumn('Outlet_Type', explode('Outlet_Type')).show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------+-----------------+----+----------+----------+-----------+--------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|Outlet_Type|Item_Outlet_Sales|flag| curr_date|week_after|week_before|datediff|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------+-----------------+----+----------+----------+-----------+--------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket|         3735.138| new|2026-07-25|2026-08-01| 18-07-2026|       7|
|       

In [65]:
df_exp.withColumn('type1_flag', array_contains('Outlet_Type', 'Type1')).show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+--------------------+-----------------+----+----------+----------+-----------+--------+----------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|         Outlet_Type|Item_Outlet_Sales|flag| curr_date|week_after|week_before|datediff|type1_flag|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+--------------------+-----------------+----+----------+----------+-----------+--------+----------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|[Supermarket, Type1]|      

### GROUP_BY

#### Scenario 1

In [66]:
df.groupBy('Item_Type').agg(sum('Item_MRP')).show()

+--------------------+------------------+
|           Item_Type|     sum(Item_MRP)|
+--------------------+------------------+
|       Starchy Foods|21880.027399999995|
|        Baking Goods| 81894.73640000001|
|              Breads| 35379.11979999999|
|Fruits and Vegeta...|178124.08099999998|
|                Meat|59449.863799999956|
|         Hard Drinks|29334.676599999995|
|         Soft Drinks|58514.164999999964|
|           Household|135976.52539999998|
|           Breakfast|        15596.6966|
|               Dairy|101276.45959999996|
|         Snack Foods|175433.92040000003|
|              Others|22451.891600000006|
|             Seafood| 9077.870000000003|
|              Canned|  90706.7269999999|
|        Frozen Foods|118558.88140000001|
|  Health and Hygiene|        68025.8388|
+--------------------+------------------+



#### Scenario 2

In [67]:
df.groupBy('Item_Type').agg(avg('Item_MRP')).show()

+--------------------+------------------+
|           Item_Type|     avg(Item_MRP)|
+--------------------+------------------+
|       Starchy Foods|147.83802297297294|
|        Baking Goods|126.38076604938273|
|              Breads| 140.9526685258964|
|Fruits and Vegeta...|144.58123457792206|
|                Meat|139.88203247058814|
|         Hard Drinks|137.07792803738315|
|         Soft Drinks|131.49250561797746|
|           Household|149.42475318681318|
|           Breakfast|141.78815090909092|
|               Dairy|148.49920762463336|
|         Snack Foods|146.19493366666669|
|              Others|132.85142958579885|
|             Seafood|141.84171875000004|
|              Canned|139.76383204930647|
|        Frozen Foods|138.50336612149533|
|  Health and Hygiene|130.81892076923077|
+--------------------+------------------+



#### Scenario 3

In [68]:
df.groupBy('Item_Type', 'Outlet_Size').agg(sum('Item_MRP').alias('Total_MRP')).show()

+--------------------+-----------+------------------+
|           Item_Type|Outlet_Size|         Total_MRP|
+--------------------+-----------+------------------+
|       Starchy Foods|     Medium| 7124.136199999997|
|Fruits and Vegeta...|     Medium|59047.217200000014|
|       Starchy Foods|       NULL|         6040.6402|
|              Breads|       NULL|        10011.5004|
|        Baking Goods|       NULL|23433.838799999994|
|Fruits and Vegeta...|       NULL|49758.730999999985|
|        Frozen Foods|       High|12588.291000000001|
|         Soft Drinks|       High| 6456.165199999999|
|           Breakfast|      Small|3917.0407999999998|
|                Meat|     Medium| 20326.45059999999|
|Fruits and Vegeta...|       High| 20671.34759999999|
|                Meat|       High| 5627.036400000002|
|        Baking Goods|       High| 9431.749199999998|
|           Household|     Medium| 42688.57439999998|
|                Meat|       NULL|16158.166000000005|
|         Hard Drinks|      

#### Scenario 4

In [69]:
df.groupBy('Item_Type', 'Outlet_Size').agg(sum('Item_MRP'), avg('Item_MRP')).show()

+--------------------+-----------+------------------+------------------+
|           Item_Type|Outlet_Size|     sum(Item_MRP)|     avg(Item_MRP)|
+--------------------+-----------+------------------+------------------+
|       Starchy Foods|     Medium| 7124.136199999997| 148.4195041666666|
|Fruits and Vegeta...|     Medium|59047.217200000014| 142.9714702179177|
|       Starchy Foods|       NULL|         6040.6402|140.48000465116277|
|              Breads|       NULL|        10011.5004|139.04861666666667|
|        Baking Goods|       NULL|23433.838799999994|126.66939891891889|
|Fruits and Vegeta...|       NULL|49758.730999999985|142.57516045845267|
|        Frozen Foods|       High|12588.291000000001|         136.82925|
|         Soft Drinks|       High| 6456.165199999999|131.75847346938772|
|           Breakfast|      Small|3917.0407999999998|130.56802666666667|
|                Meat|     Medium| 20326.45059999999|136.41913154362408|
|Fruits and Vegeta...|       High| 20671.3475999999

### COLLECT_LIST

In [70]:
data = [('user1','book1'),
        ('user1','book2'),
        ('user2','book2'),
        ('user2','book4'),
        ('user3','book1')]

schema = 'user string, book string'

df_book = spark.createDataFrame(data,schema)

df_book.show()

+-----+-----+
| user| book|
+-----+-----+
|user1|book1|
|user1|book2|
|user2|book2|
|user2|book4|
|user3|book1|
+-----+-----+



In [71]:
df_book.groupBy('user').agg(collect_list('book')).show()

+-----+------------------+
| user|collect_list(book)|
+-----+------------------+
|user1|    [book1, book2]|
|user3|           [book1]|
|user2|    [book2, book4]|
+-----+------------------+



### PIVOT

In [72]:
df.groupBy('Item_Type').pivot('Outlet_Size').agg(avg('Item_MRP')).show()

+--------------------+------------------+------------------+------------------+------------------+
|           Item_Type|              null|              High|            Medium|             Small|
+--------------------+------------------+------------------+------------------+------------------+
|       Starchy Foods|140.48000465116277|158.15707368421053| 148.4195041666666| 150.2701736842105|
|              Breads|139.04861666666667|         133.75896| 140.8610385542169| 145.5236507042254|
|        Baking Goods|126.66939891891889|129.20204383561642|126.17856847290639|125.21336363636368|
|Fruits and Vegeta...|142.57516045845267|145.57287042253515| 142.9714702179177|148.31336951219507|
|                Meat|139.29453448275865| 137.2447902439025|136.41913154362408|145.69925042016808|
|         Hard Drinks| 134.3875333333333| 141.9275217391304|142.83769599999994|        129.758784|
|         Soft Drinks|133.42344360902257|131.75847346938772| 128.2696817518248| 132.8550428571429|
|         

### When-Otherwise

#### Scenario 1

In [73]:
df.withColumn('veg_flag', when(col('Item_Type')=='Meat', 'Non-Veg').otherwise('Veg')).show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+--------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag| curr_date|week_after|week_before|datediff|veg_flag|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+--------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138| new|2

#### Scenario 2

In [74]:
df.withColumn('veg_exp_flag', when((col('Item_Type')!='Meat') & (col('Item_MRP')<100.0), 'Veg_Inexpensive') \
                              .when((col('Item_Type')!='Meat') & (col('Item_MRP')>=100.0), 'Veg_Expensive') \
                                .otherwise('Non_Veg')).show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+---------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag| curr_date|week_after|week_before|datediff|   veg_exp_flag|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+---------------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|   

### JOINS

In [75]:
dataj1 = [('1','gaur','d01'),
          ('2','kit','d02'),
          ('3','sam','d03'),
          ('4','tim','d03'),
          ('5','aman','d05'),
          ('6','nad','d06')] 

schemaj1 = 'emp_id STRING, emp_name STRING, dept_id STRING' 

df1 = spark.createDataFrame(dataj1,schemaj1)

dataj2 = [('d01','HR'),
          ('d02','Marketing'),
          ('d03','Accounts'),
          ('d04','IT'),
          ('d05','Finance')]

schemaj2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(dataj2,schemaj2)

In [76]:
df1.show()

+------+--------+-------+
|emp_id|emp_name|dept_id|
+------+--------+-------+
|     1|    gaur|    d01|
|     2|     kit|    d02|
|     3|     sam|    d03|
|     4|     tim|    d03|
|     5|    aman|    d05|
|     6|     nad|    d06|
+------+--------+-------+



In [77]:
df2.show()

+-------+----------+
|dept_id|department|
+-------+----------+
|    d01|        HR|
|    d02| Marketing|
|    d03|  Accounts|
|    d04|        IT|
|    d05|   Finance|
+-------+----------+



#### Inner Join

In [78]:
df1.join(df2, df1['dept_id']==df2['dept_id'], 'inner').show()

+------+--------+-------+-------+----------+
|emp_id|emp_name|dept_id|dept_id|department|
+------+--------+-------+-------+----------+
|     1|    gaur|    d01|    d01|        HR|
|     2|     kit|    d02|    d02| Marketing|
|     3|     sam|    d03|    d03|  Accounts|
|     4|     tim|    d03|    d03|  Accounts|
|     5|    aman|    d05|    d05|   Finance|
+------+--------+-------+-------+----------+



#### Left Join

In [79]:
df1.join(df2, df1['dept_id']==df2['dept_id'], 'left').orderBy('emp_id', ascending=True).show()

+------+--------+-------+-------+----------+
|emp_id|emp_name|dept_id|dept_id|department|
+------+--------+-------+-------+----------+
|     1|    gaur|    d01|    d01|        HR|
|     2|     kit|    d02|    d02| Marketing|
|     3|     sam|    d03|    d03|  Accounts|
|     4|     tim|    d03|    d03|  Accounts|
|     5|    aman|    d05|    d05|   Finance|
|     6|     nad|    d06|   NULL|      NULL|
+------+--------+-------+-------+----------+



#### Right Join

In [80]:
df1.join(df2, df1['dept_id']==df2['dept_id'], 'right').orderBy('emp_id', ascending=True).show()

+------+--------+-------+-------+----------+
|emp_id|emp_name|dept_id|dept_id|department|
+------+--------+-------+-------+----------+
|  NULL|    NULL|   NULL|    d04|        IT|
|     1|    gaur|    d01|    d01|        HR|
|     2|     kit|    d02|    d02| Marketing|
|     3|     sam|    d03|    d03|  Accounts|
|     4|     tim|    d03|    d03|  Accounts|
|     5|    aman|    d05|    d05|   Finance|
+------+--------+-------+-------+----------+



#### Anti Join

In [81]:
df1.join(df2, df1['dept_id']==df2['dept_id'], 'anti').orderBy('emp_id', ascending=True).show()

+------+--------+-------+
|emp_id|emp_name|dept_id|
+------+--------+-------+
|     6|     nad|    d06|
+------+--------+-------+



### ROW_NUMBER()

In [82]:
from pyspark.sql.window import Window

In [83]:
df.withColumn('rowCol', row_number().over(Window.orderBy('Item_Identifier'))).show()

26/07/25 02:36:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/25 02:36:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/25 02:36:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---------------+-----------+----------------+---------------+-----------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|  Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag| curr_date|week_after|week_before|datediff|rowCol|
+---------------+-----------+----------------+---------------+-----------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+------+
|          DRA12|       11.6|         Low Fat|    0.041177505|Soft Drinks|140.3154|           OUT017|                     2007|       NULL|              Tier 2|Supermarket Type1|        2552.6772| new|2026-07-25|2026-08-01| 18-07-2026|       7|

### RANK vs. DENSE_RANK

In [84]:
df.withColumn('rank', rank().over(Window.orderBy('Item_Identifier'))).show()

26/07/25 02:36:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/25 02:36:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/25 02:36:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---------------+-----------+----------------+---------------+-----------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+----+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|  Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag| curr_date|week_after|week_before|datediff|rank|
+---------------+-----------+----------------+---------------+-----------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+----+
|          DRA12|       11.6|         Low Fat|    0.041177505|Soft Drinks|140.3154|           OUT017|                     2007|       NULL|              Tier 2|Supermarket Type1|        2552.6772| new|2026-07-25|2026-08-01| 18-07-2026|       7|   1|


In [85]:
df.withColumn('dense_rank', dense_rank().over(Window.orderBy('Item_Identifier'))).show()

26/07/25 02:36:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/25 02:36:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/25 02:36:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---------------+-----------+----------------+---------------+-----------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+----------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|  Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag| curr_date|week_after|week_before|datediff|dense_rank|
+---------------+-----------+----------------+---------------+-----------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+----------+
|          DRA12|       11.6|         Low Fat|    0.041177505|Soft Drinks|140.3154|           OUT017|                     2007|       NULL|              Tier 2|Supermarket Type1|        2552.6772| new|2026-07-25|2026-08-01| 18-07-20

In [86]:
df.withColumn('dense_rank', dense_rank().over(Window.orderBy(col('Item_Identifier').desc()))).show()

26/07/25 02:36:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/25 02:36:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/25 02:36:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---------------+-----------+----------------+---------------+------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+----------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|         Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag| curr_date|week_after|week_before|datediff|dense_rank|
+---------------+-----------+----------------+---------------+------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+----------+
|          NCZ54|      14.65|         Low Fat|    0.083359391|         Household|161.9552|           OUT046|                     1997|      Small|              Tier 1|Supermarket Type1|        4711.2008| new|202

### Cumulative Sum

In [87]:
# Prints sum of MRP for each item type (but sums over previous type as well)
df.withColumn('cumsum', sum('Item_MRP').over(Window.orderBy('Item_Type'))).show()

26/07/25 02:36:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/25 02:36:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/25 02:36:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/25 02:36:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/25 02:36:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---------------+-----------+----------------+---------------+------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+-----------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|   Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag| curr_date|week_after|week_before|datediff|           cumsum|
+---------------+-----------+----------------+---------------+------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+-----------------+
|          FDP36|     10.395|         Regular|              0|Baking Goods| 51.4008|           OUT018|                     2009|     Medium|              Tier 3|Supermarket Type2|         556.6088| new|2026-0

In [88]:
df.withColumn('cumsum', sum('Item_MRP').over(Window.orderBy('Item_Type') \
                                             .rowsBetween(Window.unboundedPreceding, Window.currentRow))).show()

26/07/25 02:36:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/25 02:36:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/25 02:36:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/25 02:36:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/25 02:36:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---------------+-----------+----------------+---------------+------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+------------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|   Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag| curr_date|week_after|week_before|datediff|            cumsum|
+---------------+-----------+----------------+---------------+------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+------------------+
|          FDP36|     10.395|         Regular|              0|Baking Goods| 51.4008|           OUT018|                     2009|     Medium|              Tier 3|Supermarket Type2|         556.6088| new|202

# Try not to use User defined functions : Not optimized by PySpark engines

### USER DEFINED FUNCTIONS (UDF)

#### STEP 1

In [89]:
def my_func(x):
    return x*x

#### STEP 2

In [90]:
my_udf = udf(my_func)

/opt/spark/python/pyspark/sql/udf.py:120: RuntimeWarning: Arrow optimization failed to enable because PyArrow or Pandas is not installed. Falling back to a non-Arrow-optimized UDF.
  warnings.warn(


In [91]:
df = df.withColumn('Item_MRP', col('Item_MRP').cast(DoubleType()))

In [92]:
df.withColumn('mynewcol', my_udf('Item_MRP')).show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+------------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag| curr_date|week_after|week_before|datediff|          mynewcol|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+------------------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket 

### DATA WRITING

#### CSV

In [93]:
df.write.format('csv').mode('overwrite').save('/opt/spark-apps/CSV/data.csv')

In [94]:
# Modes
# append - add to the existing file
# overwrite - overwrite the existing file
# error - if the file is already there it will raise an error and do nothing
# ignore - if the file is already there it will do nothing (silent fail)

#### APPEND

In [95]:
df.write.format('csv') \
        .mode('append') \
        .save('/opt/spark-apps/CSV/data.csv')

In [96]:
df.write.format('csv') \
        .mode('append') \
        .option('path', '/opt/spark-apps/CSV/data.csv') \
        .save()

#### Overwrite

In [97]:
df.write.format('csv') \
        .mode('overwrite') \
        .option('path', '/opt/spark-apps/CSV/data.csv') \
        .save()

#### Error

In [100]:
# This one will raise error so commented out
"""
df.write.format('csv') \
        .mode('error') \
        .option('path', '/opt/spark-apps/CSV/data.csv') \
        .save()
"""

"\ndf.write.format('csv')         .mode('error')         .option('path', '/opt/spark-apps/CSV/data.csv')         .save()\n"

#### Ignore

In [102]:
df.write.format('csv') \
        .mode('ignore') \
        .option('path', '/opt/spark-apps/CSV/data.csv') \
        .save()

In [103]:
# Parquet
# Columnar format - each column is saved as each row
# To query only the selected columns need to be retrieved

#### PARQUET

In [104]:
df.write.format('parquet') \
        .mode('overwrite') \
        .option('path', '/opt/spark-apps/CSV/data.csv') \
        .save()

In [105]:
# Parquet vs. Delta Lake
# Delta Lake is built on top of parquet, with separate file
# that holds transactions history and metadata
# It supports ACID transaction

#### TABLE

In [106]:
df.write.format('parquet') \
        .mode('overwrite') \
        .saveAsTable('my_table')

In [107]:
# Managed Table vs. External Table

# Managed Table - managed by Databricks or Spark engine
# Dropping table gets the table deleted

# External Table - data is stored in custom location indicated by user in table
# Dropping table only deletes metadata (managed by Spark) - data is intact

### SparkSQL

In [108]:
# No difference in performance between Spark operations and SparkSQL

#### createTempView

In [109]:
df.createTempView('my_view')

In [114]:
spark.sql('SELECT * FROM my_view WHERE Item_Weight BETWEEN 9.0 AND 10.0').show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag| curr_date|week_after|week_before|datediff|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138| new|2026-07-25|2026-08-01| 18-07

In [115]:
df_sql = spark.sql("SELECT * FROM my_view WHERE Item_Fat_Content = 'LF'")

In [116]:
df_sql.show()

+---------------+-----------+----------------+---------------+------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|         Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|flag| curr_date|week_after|week_before|datediff|
+---------------+-----------+----------------+---------------+------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----+----------+----------+-----------+--------+
|          FDM39|       6.42|              LF|    0.089498926|             Dairy|178.1002|           OUT010|                     1998|       NULL|              Tier 3|    Grocery Store|         358.2004| new|2026-07-25|2026-08-01| 18-07-2026|  